In [75]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/merged_219.csv")

qdf = pd.read_excel('../data/2024EMA_q0.xlsx', 'all responses', header=[0,1])
qdf.columns = qdf.columns.droplevel(0)
qdf = qdf.iloc[:,4:]
third_row = qdf.iloc[0, :]
response = third_row[third_row == '응답'].index
qdf = qdf.drop(columns = response)
qdf = qdf[1:].reset_index(drop=True)

phq_df=qdf.iloc[:,:15]

old_cols = [
    '문자로 전달받은 본인의 참여코드 ID를 입력해주세요.(예시 : Q001)',
    '정신건강의학과 전문의를 만나 도움을 받으신 경험이 있으신가요?.1',
    '심리 상담을 받은 경험이 있으신가요?.1',
    '약물 치료를 받으신 경험이 있으신가요?.1',
    "정신건강의학과 치료 경험이 '있다'라고 표시 했을 경우, 주호소 증상 또는 진단명을 적어주세요.",
    '현재 처방 받고 계신 정신건강의학과 약물을 작성해주세요. (만약 연구 기간 중 중단하게 된다면 연구진에게 알려주셔야 합니다.)',
    '기분이 가라앉거나, 우울하거나 희망이 없다고 느꼈다..1',
    '평소 하던 일에 대한 흥미가 없어지거나 즐거움을 느끼지 못했다..1',
    '잠들기가 어렵거나 자주 깼다. 혹은 너무 많이 잤다..1',
    '평소보다 식욕이 줄었다. 혹은 평소보다 많이 먹었다..1',
    '다른 사람들이 눈치 챌 정도로 평소보다 말과 행동이 느려졌다. 혹은 너무 안절부절 못해서 가만히 앉아 있을 수 없었다..1',
    '피곤하고 기운이 없었다..1',
    '내가 잘못했거나 실패했다는 생각이 들었다. 혹은 자식과 가족을 실망시켰다고 생각했다..1',
    '학교 공부, 독서, TV 시청과 같은 일상적인 일에도 집중할 수가 없었다..1',
    '차라리 죽는 것이 더 낫겠다고 생각했다. 혹은 자해할 생각을 했다..1'
]

new_names = [
    "Qid",
    "treatment",
    "counseling",
    "medication",
    "treatment_detail",
    "medication_current",
    
    "depressed",
    "anhedonia",
    "sleep",
    "appetite_changes",
    "moving",
    "fatigue",
    "failure",
    "concentrating",
    "self-harm"
]

rename_map = dict(zip(old_cols, new_names))
phq_df = phq_df.rename(columns=rename_map)

demo = pd.read_csv('../data/participants_219명.csv')
demo = demo[["Qid", "id"]]
phq_df = pd.merge(demo, phq_df, on='Qid', how="left").drop(columns=["Qid"]).sort_values("id").reset_index(drop=True)
# phq_df.loc[phq_df["medication_current"].str.contains("없음", na=False), "medication_current"] = np.nan
phq_df.loc[phq_df["medication_current"].isin(["없습니다", "없음", "X", "."]), "medication_current"] = np.nan
phq_df.loc[phq_df["treatment_detail"].isin(["없습니다", "없음", "X", "."]), "treatment_detail"] = np.nan


phq_df["somantic"] = phq_df[["sleep", "fatigue", "appetite_changes", "concentrating", "moving"]].sum(axis=1)
phq_df["cognitive"] = phq_df[["anhedonia", "depressed", "failure", "self-harm"]].sum(axis=1)
phq_df["depression_diagnosis"] = phq_df["treatment_detail"].str.contains("우울", na=False)


somatic_higher = (phq_df["somantic"] > phq_df["cognitive"]).sum()
cognitive_higher = (phq_df["somantic"] < phq_df["cognitive"]).sum()
equal_count = (phq_df["somantic"] == phq_df["cognitive"]).sum()
print(f"somantic이 더 높은 사람 수: {somatic_higher}")
print(f"cognitive가 더 높은 사람 수: {cognitive_higher}")
print(f"같은 사람 수: {equal_count}")

print(f"우울 진단 수: {(phq_df['depression_diagnosis'] == True).sum()}")
depression_medicine = ((phq_df["depression_diagnosis"] == True) & (phq_df["medication_current"].notna())).sum()
print(f"우울 진단이면서 약 복용 중인 사람 수: {depression_medicine}")
print(f"약물 복용 중인 사람 수: {phq_df['medication_current'].notna().sum()}")


phq_df.to_csv("../data/phq_subgroup.csv")

somantic이 더 높은 사람 수: 185
cognitive가 더 높은 사람 수: 12
같은 사람 수: 22
우울 진단 수: 29
우울 진단이면서 약 복용 중인 사람 수: 12
약물 복용 중인 사람 수: 21


In [77]:
phq_df_delivered = pd.DataFrame({
    "id": phq_df["id"],
    "somatic_high": phq_df["somantic"] > phq_df["cognitive"],
    "depression_diagnosis": phq_df["depression_diagnosis"],
    "depression_medicine": (phq_df["depression_diagnosis"] == True) & (phq_df["medication_current"].notna())
})

phq_df_delivered.to_csv("../data/phq_subgroup_delivered.csv")